# [실습 9] Ko-Ultrafeedback 으로 강화학습(DPO)

**목표** — [`maywell/ko_Ultrafeedback_binarized`](https://huggingface.co/datasets/maywell/ko_Ultrafeedback_binarized) 의 한국어 **선호도 쌍** 데이터를 사용해, 모델이 "좋은 답변" 쪽으로 정렬되도록 **DPO(Direct Preference Optimization)** 학습을 수행합니다.

## 왜 이게 "강화학습" 인가?

고전 RLHF는 (1) 보상모델을 학습하고 (2) PPO 로 정책을 업데이트했습니다.  
**DPO** 는 이 두 단계를 닫힌 형태(closed-form)로 합쳐, 선호도 데이터만 있으면 정책을 직접 업데이트합니다.

$$\mathcal{L}_{\text{DPO}}(\theta) = -\mathbb{E}_{(x,y_w,y_l)} \log \sigma\!\Big(\beta \big[\log\tfrac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \log\tfrac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\big]\Big)$$

- $y_w$ = chosen (선호되는 답), $y_l$ = rejected (덜 선호되는 답)
- $\pi_{\text{ref}}$ = SFT 직후의 고정된 정책(기준점)
- $\beta$ = KL 페널티 강도

직관: **좋은 답의 확률은 올리고, 나쁜 답의 확률은 내리되, 너무 벗어나지 않도록 ref 와 KL 거리를 유지**.

| 항목 | 값 |
|---|---|
| 데이터셋 | `maywell/ko_Ultrafeedback_binarized` |
| 베이스 | [실습8]에서 학습한 KoAlpaca SFT 어댑터 (없어도 자동 폴백) |
| 방법 | DPO + LoRA |
| 권장 환경 | Colab T4 GPU |

## 0. 환경 준비

In [ ]:
!pip install -q --upgrade \
    transformers==4.44.2 datasets==2.21.0 accelerate==0.34.2 \
    peft==0.12.0 trl==0.10.1 bitsandbytes==0.43.3 sentencepiece

In [ ]:
import os, torch
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
print("torch :", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU   :", torch.cuda.get_device_name(0))

---
## 1. 데이터셋 로드 & 탐색 — `maywell/ko_Ultrafeedback_binarized`

OpenBMB 의 UltraFeedback 을 한국어로 번역·정제한 데이터셋입니다.  
DPO 학습에 바로 쓸 수 있도록 **`prompt / chosen / rejected`** 형태로 binarize 되어 있습니다.

In [ ]:
from datasets import load_dataset

# 학습 시간 단축을 위해 일부만 — 전체로 돌릴 때는 split="train" 그대로
raw = load_dataset("maywell/ko_Ultrafeedback_binarized", split="train")
print("전체 샘플 수:", len(raw))
print("컬럼       :", raw.column_names)
print("\n--- 첫 샘플 ---")
for k in raw.column_names:
    v = raw[0][k]
    print(f"{k:>10s}: {str(v)[:200]}{'...' if len(str(v))>200 else ''}")

In [ ]:
# 두 응답이 얼마나 다른지 / 어떤 식의 선호 신호인지 직접 보기
for i in [0, 1, 2]:
    s = raw[i]
    print(f"========== 샘플 {i} ==========")
    print("[PROMPT]  ", str(s.get("prompt", ""))[:200])
    print("[CHOSEN]  ", str(s.get("chosen", ""))[:200])
    print("[REJECTED]", str(s.get("rejected", ""))[:200])
    print()

### 데이터 정제

TRL의 `DPOTrainer` 가 기대하는 컬럼은 **`prompt / chosen / rejected`** 세 개. 그 외 컬럼은 제거하고, 빈 문자열 / 너무 긴 샘플은 걸러냅니다.

In [ ]:
# chosen/rejected 가 chat 메시지 리스트 형태로 저장된 데이터셋도 있어, 문자열로 정규화
def to_text(x):
    if isinstance(x, str):
        return x.strip()
    if isinstance(x, list):  # [{"role":..., "content":...}] 형태
        for m in reversed(x):
            if isinstance(m, dict) and m.get("role") == "assistant":
                return str(m.get("content", "")).strip()
        return " ".join(str(m.get("content", m)) for m in x).strip()
    return str(x).strip()

def normalize(ex):
    return {
        "prompt":   to_text(ex.get("prompt", "")),
        "chosen":   to_text(ex.get("chosen", "")),
        "rejected": to_text(ex.get("rejected", "")),
    }

clean = raw.map(normalize, remove_columns=raw.column_names)

MAX_PROMPT = 256
MAX_RESP   = 384

clean = clean.filter(lambda x:
    0 < len(x["prompt"]) <= MAX_PROMPT and
    0 < len(x["chosen"]) <= MAX_RESP and
    0 < len(x["rejected"]) <= MAX_RESP and
    x["chosen"] != x["rejected"]
)

print(f"정제 후 샘플 수: {len(clean)} / {len(raw)}")
print(clean[0])

In [ ]:
# 실습 속도를 위해 1000개만 추출 — 본격 학습 시 키울 것
train_ds = clean.shuffle(seed=42).select(range(min(1000, len(clean))))
print("학습용 샘플 수:", len(train_ds))

---
## 2. 베이스 모델 로드

이상적으로는 **[실습8] 의 SFT 어댑터**를 로드해 그 위에서 DPO 를 합니다.  
어댑터가 없으면 베이스 모델만 로드하여 진행합니다.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import os

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
SFT_ADAPTER_PATH = "./koalpaca_lora_adapter"   # [실습8] 산출물

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map="auto",
)
base.config.use_cache = False

if os.path.isdir(SFT_ADAPTER_PATH):
    print(f"[OK] SFT 어댑터 발견 → 결합 후 DPO 진행: {SFT_ADAPTER_PATH}")
    model = PeftModel.from_pretrained(base, SFT_ADAPTER_PATH, is_trainable=True)
else:
    print("[INFO] SFT 어댑터가 없어 베이스 모델에서 바로 DPO를 진행합니다.")
    model = base

### 학습 전 베이스라인 응답 기록

DPO 효과를 보려면 학습 전 응답을 먼저 찍어둡니다.

In [ ]:
EVAL_PROMPTS = [
    "매운 음식을 못 먹는 사람을 위한 점심 메뉴를 추천해 줘.",
    "머신러닝에서 과적합을 막는 방법 세 가지를 설명해 줘.",
    "초보자에게 파이썬을 추천하는 이유를 친절하게 알려 줘.",
]

@torch.no_grad()
def generate(prompt, max_new_tokens=160):
    text = f"### 명령어:\n{prompt}\n\n### 응답:\n"
    inp = tokenizer(text, return_tensors="pt").to(model.device)
    out = model.generate(**inp, max_new_tokens=max_new_tokens,
                         do_sample=True, temperature=0.7, top_p=0.9,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True)

print("=" * 60)
print("[DPO 학습 전 응답]")
print("=" * 60)
baseline = {}
for q in EVAL_PROMPTS:
    a = generate(q)
    baseline[q] = a
    print("Q:", q); print("A:", a); print("-" * 60)

---
## 3. DPO 학습 설정

주요 하이퍼파라미터:
- **`beta`**: DPO loss 의 온도. 0.1 ~ 0.5 권장. 크면 ref 에서 멀어지지 않으려는 보수적 업데이트.
- **`learning_rate`**: SFT 보다 한참 작게 (1e-6 ~ 5e-6). 너무 크면 ref 에서 급격히 이탈해 응답이 무너짐.
- **`max_length / max_prompt_length`**: 메모리와 직결. T4 면 보수적으로 잡기.

In [ ]:
from peft import LoraConfig
from transformers import TrainingArguments
from trl import DPOTrainer

dpo_lora = LoraConfig(
    r=8, lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
)

dpo_args = TrainingArguments(
    output_dir="./dpo_out",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,    # 유효 배치 = 8
    learning_rate=5e-6,
    max_steps=200,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    bf16=True,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    remove_unused_columns=False,
    report_to="none",
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,            # PEFT 모델인 경우 TRL이 어댑터 비활성화로 자동 ref 구성
    args=dpo_args,
    beta=0.1,
    train_dataset=train_ds,
    tokenizer=tokenizer,
    max_length=512,
    max_prompt_length=MAX_PROMPT,
    peft_config=dpo_lora if not isinstance(model, PeftModel) else None,
)

dpo_trainer.model.print_trainable_parameters()

---
## 4. 학습 실행 — DPO

**모니터링 포인트:**
- `rewards/chosen` > `rewards/rejected` 이어야 정상
- 둘의 격차 `rewards/margins` 가 점점 커져야 함
- `rewards/accuracies` 는 0.6 → 0.9 부근으로 상승하는 게 좋음

In [ ]:
result = dpo_trainer.train()
print("\nDPO 학습 완료")
print("최종 loss :", result.training_loss)
print("총 스텝수 :", result.global_step)

In [ ]:
SAVE_DIR = "./dpo_lora_adapter"
dpo_trainer.model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("저장 경로:", SAVE_DIR)

---
## 5. DPO 학습 곡선 시각화

DPO 로그에는 일반 `loss` 외에도 RLHF 특유의 보상 지표가 함께 찍힙니다.

In [ ]:
import matplotlib.pyplot as plt

hist = dpo_trainer.state.log_history
def pull(key):
    xs = [h["step"] for h in hist if key in h]
    ys = [h[key]    for h in hist if key in h]
    return xs, ys

fig, axes = plt.subplots(1, 3, figsize=(14, 3.2))

x, y = pull("loss")
axes[0].plot(x, y, marker="o", linewidth=1)
axes[0].set_title("DPO loss"); axes[0].grid(alpha=0.3)

xc, yc = pull("rewards/chosen")
xr, yr = pull("rewards/rejected")
if xc and xr:
    axes[1].plot(xc, yc, label="chosen", marker="o", linewidth=1)
    axes[1].plot(xr, yr, label="rejected", marker="o", linewidth=1)
axes[1].set_title("rewards/chosen vs rejected"); axes[1].legend(); axes[1].grid(alpha=0.3)

xa, ya = pull("rewards/accuracies")
if xa:
    axes[2].plot(xa, ya, marker="o", linewidth=1, color="green")
axes[2].set_title("rewards/accuracies"); axes[2].set_ylim(0, 1); axes[2].grid(alpha=0.3)

for ax in axes: ax.set_xlabel("step")
plt.tight_layout(); plt.show()

---
## 6. 학습 후 응답 — 베이스라인 vs DPO

DPO 가 정상 동작했다면, 일반적으로:
- 답이 더 **길고 구체적**으로 변하거나
- **목록·구조화** 가 더 잘 되거나
- **거절·회피 표현이 줄어드는** 경향이 보입니다.

(데모 스텝 수가 작으면 차이가 미미할 수 있으니, 본격 학습은 `max_steps` 을 1000+ 로 늘리세요.)

In [ ]:
print("=" * 60)
print("[DPO 학습 후 응답]")
print("=" * 60)
for q in EVAL_PROMPTS:
    a_after = generate(q)
    print("Q:", q)
    print("BEFORE:", baseline[q])
    print("AFTER :", a_after)
    print("-" * 60)

---
## 7. 보상 모델 없이 학습이 가능한 이유 (요약)

Bradley–Terry 선호도 모델 아래에서, 최적 정책의 보상 함수는

$$r^*(x, y) = \beta \log \frac{\pi^*(y|x)}{\pi_{\text{ref}}(y|x)} + Z(x)$$

로 **현재 정책과 ref 정책의 로그비** 로 정확히 표현됩니다.  
→ 그러므로 보상 모델을 따로 두지 않고, 정책 자체의 로그확률 차이로 보상을 대체할 수 있습니다.  
→ 이게 PPO 의 보상-가치-정책 3중 구조를 **하나의 supervised-style 손실** 로 압축한 DPO 의 정체.

## 8. 응용 과제

- 🔧 **A. `beta` 비교**: 0.05 / 0.1 / 0.3 으로 각각 학습 후 응답 다양성/안정성을 비교
- 🔧 **B. 데이터 비율**: train_ds 크기를 500 / 2000 / 전체로 늘릴 때 `rewards/margins` 변화 관찰
- 🔧 **C. ORPO 비교**: `trl.ORPOTrainer` 로 같은 데이터셋을 학습해 보고 DPO 와 응답을 비교
- 🔧 **D. 평가**: `EQ-Bench-Ko` / `KoBest` 등으로 정성 평가에서 정량 평가로 확장

## 참고
- 논문: [Direct Preference Optimization (Rafailov et al., 2023)](https://arxiv.org/abs/2305.18290)
- TRL DPO 가이드: <https://huggingface.co/docs/trl/dpo_trainer>
- 데이터셋 페이지: <https://huggingface.co/datasets/maywell/ko_Ultrafeedback_binarized>